# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All entities such as record sets, fields, and columns are referenced by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Note: dataset.metadata is a dataclass, access fields as attributes
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

# Show available top-level metadata fields
print(f"Available metadata attributes: {dir(dataset.metadata)}")

## 2. Data Overview

List available record sets and their fields. All identifiers are referenced by `@id` as per the FAIR<sup>2</sup> schema. We'll also print a sample record for each set.

In [ ]:
# List record sets by `@id`
record_set_ids = [rset['@id'] for rset in dataset.metadata.record_sets]
print("Record set @id's found:")
for rid in record_set_ids:
    print(f"  - {rid}")

# For each record set, list field @ids and show one sample record
for rid in record_set_ids:
    print(f"\nRecord set: {rid}")
    fields = next(rs for rs in dataset.metadata.record_sets if rs['@id'] == rid)['fields']
    field_ids = [field['@id'] for field in fields]
    print(f"Fields: {field_ids}")
    # Print a sample record
    records = list(dataset.records(record_set=rid))
    if records:
        print(f"Sample record from {rid}:\n{json.dumps(records[0], indent=2)}")
    else:
        print("No records found.")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for further analysis, using the record set and field `@id`s from the previous step. DataFrames are stored by the record set `@id`.

In [ ]:
# Extract data from each record set into a dictionary of DataFrames
dataframes = {}
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"Loaded {len(df)} records for record set '{rid}'. Columns:")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print(f"No records found for record set '{rid}'.")

## 4. Exploratory Data Analysis (EDA)

We will analyze one record set and perform basic cleaning and transformation steps. We reference all fields by their `@id`.

First, choose a record set with substantial numeric data (e.g., protein abundance, coverage, or peptide count fields). Identify a numeric field to filter and normalize, and a group-by field for aggregation.

In [ ]:
# --- You may change these depending on the actual field @ids in the dataset ---
# Select a record set for deeper analysis
eda_record_set_id = record_set_ids[0]  # e.g., the first one
df = dataframes[eda_record_set_id]
print(f"Analyzing record set: {eda_record_set_id}")

# List all columns (should be field @ids)
print("Field (column) @id's:")
print(df.columns.tolist())

# Choose a numeric field
numeric_field_id = None
for col in df.columns:
    # Heuristically pick a column that is numeric
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try to cast possible numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue

print(f"Selected numeric field: {numeric_field_id}")

# Filter and normalize
if numeric_field_id:
    # Remove NaN values
    subset = df.dropna(subset=[numeric_field_id])
    threshold = subset[numeric_field_id].mean()  # Use mean as filter threshold
    filtered_df = subset[subset[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean): {len(filtered_df)} records")

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field - e.g., second string column
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_string_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (showing means of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and its normalized values after filtering. This helps understand the data spread and outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and len(filtered_df) > 0:
    plt.figure(figsize=(10,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} after filtering")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(10,4))
    sns.boxplot(x=filtered_df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id} (filtered)")
    plt.show()

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True)
        plt.title(f"Normalized {numeric_field_id} distribution (filtered)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion

In this notebook, we demonstrated loading, exploring, and processing the FAIR<sup>2</sup> dataset using the `mlcroissant` library. Each entity (record set, field, column) was referenced by its `@id`, ensuring traceable and standards-based data exploration.

- We overviewed metadata and printed available record sets and field IDs.
- We loaded records into pandas DataFrames and selected numeric fields for analysis.
- Standard preprocessing steps like filtering and normalization were demonstrated, with grouping and basic statistics.
- Data distributions were visualized to reveal patterns and outliers.

For more advanced analysis, consult the full [mlcroissant documentation](https://github.com/mlcommons/croissant) and schema definitions for available entities and relationships.